# ESM3 Debugging Notebook


In [1]:
import torch
from esm.pretrained import ESM3_sm_open_v0
from transformer_lens import HookedESM3, SupportedESM3Config
from esm.tokenization import get_esm3_model_tokenizers

device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "esm3"

config = SupportedESM3Config(
    use_attn_result=False,
    use_split_qkv_input=False,
    use_hook_mlp_in=False,
    use_attn_in=False,
    esm3_use_torch_layer_norm=True,
    esm3_use_torch_attention_calc=True,
    esm3_use_org_rotary=True,
    esm3_capture_activations_before_normalization=False,
    esm3_output_type="sequence"
)

print(f"Device: {device}, Model: {model_name}")


Device: cuda, Model: esm3


In [2]:
# Tokenize - ESM3 needs sequence tokens
tokenizer = get_esm3_model_tokenizers()
sequence = "MMPISAAHKIMQTSDETYTTVGNIKVKCEEVARSTLQIPGSVNALSEKCEEVARSSLPIPGSVNALSENLLWCWLEGSKGLRSSNKIKGEIYGRRTAGIDIGLANADKVHKAPKAMRIRGASREFHSVCVMPQ"

# ESM3 tokenization
encoded = tokenizer.sequence.encode(sequence)
sequence_tokens = torch.tensor(encoded, dtype=torch.int64).to(device).unsqueeze(0)
print(f"Sequence tokens shape: {sequence_tokens.shape}")


Fetching 22 files:   0%|          | 0/22 [00:00<?, ?it/s]

Sequence tokens shape: torch.Size([1, 135])


In [3]:
# Load both models
device_obj = torch.device(device)
esm3_original = ESM3_sm_open_v0(device_obj).to(device).to(torch.float32).eval()  # type: ignore[arg-type]
esm3_hooked = HookedESM3.from_pretrained(esm_cfg=config, device=device).eval()
print("Both models loaded")


/home/galkesten/miniconda3/envs/transformer_lens_cuda12_4/lib/python3.10/site-packages/esm/pretrained.py:105: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torc

Moving model to device:  cuda
Loaded pretrained model esm3_sm_open_v1 into HookedESM3
Both models loaded


In [4]:
# Register hooks for first block
hooks = {'orig_attn': {}, 'orig_ffn': {}, 'orig_ln_qkv': {}, 
         'orig_rotary_q_in': {}, 'orig_rotary_k_in': {}, 'orig_rotary_q_out': {}, 'orig_rotary_k_out': {},
         'orig_q_ln_in': {}, 'orig_q_ln_out': {}, 'orig_k_ln_in': {}, 'orig_k_ln_out': {},
         'hook_attn': {}, 'hook_mlp': {}, 'hook_ln1': {},
         'hook_rotary_q_in': {}, 'hook_rotary_k_in': {}, 'hook_rotary_q_out': {}, 'hook_rotary_k_out': {},
         'hook_q_ln_in': {}, 'hook_q_ln_out': {}, 'hook_k_ln_in': {}, 'hook_k_ln_out': {}}

def make_hook(key): return lambda m, i, o: hooks[key].update({'output': o.detach().clone()})

def make_ln_hook(key_in, key_out):
    def hook(m, i, o):
        if isinstance(i, tuple):
            hooks[key_in].update({'output': i[0].detach().clone()})
        else:
            hooks[key_in].update({'output': i.detach().clone()})
        hooks[key_out].update({'output': o.detach().clone()})
    return hook

def make_rotary_hook(key_q_in, key_k_in, key_q_out, key_k_out): 
    def hook(m, i, o):
        if isinstance(i, tuple) and len(i) >= 2:
            hooks[key_q_in].update({'output': i[0].detach().clone()})
            hooks[key_k_in].update({'output': i[1].detach().clone()})
        if isinstance(o, tuple) and len(o) >= 2:
            hooks[key_q_out].update({'output': o[0].detach().clone()})
            hooks[key_k_out].update({'output': o[1].detach().clone()})
    return hook

h1 = esm3_original.transformer.blocks[0].attn.out_proj.register_forward_hook(make_hook('orig_attn'))
h2 = esm3_original.transformer.blocks[0].ffn.register_forward_hook(make_hook('orig_ffn'))
h3 = esm3_original.transformer.blocks[0].attn.layernorm_qkv[0].register_forward_hook(make_hook('orig_ln_qkv'))
h4 = esm3_original.transformer.blocks[0].attn.rotary.register_forward_hook(
    make_rotary_hook('orig_rotary_q_in', 'orig_rotary_k_in', 'orig_rotary_q_out', 'orig_rotary_k_out'))
h5 = esm3_original.transformer.blocks[0].attn.q_ln.register_forward_hook(
    make_ln_hook('orig_q_ln_in', 'orig_q_ln_out'))
h6_orig = esm3_original.transformer.blocks[0].attn.k_ln.register_forward_hook(
    make_ln_hook('orig_k_ln_in', 'orig_k_ln_out'))
h7 = esm3_hooked.blocks[0].attn.register_forward_hook(make_hook('hook_attn'))
h8 = esm3_hooked.blocks[0].mlp.register_forward_hook(make_hook('hook_mlp'))
h9 = esm3_hooked.blocks[0].ln1.register_forward_hook(make_hook('hook_ln1'))
h10 = esm3_hooked.blocks[0].attn.rotary.register_forward_hook(
    make_rotary_hook('hook_rotary_q_in', 'hook_rotary_k_in', 'hook_rotary_q_out', 'hook_rotary_k_out'))
h11 = esm3_hooked.blocks[0].attn.q_ln.register_forward_hook(
    make_ln_hook('hook_q_ln_in', 'hook_q_ln_out'))
h12 = esm3_hooked.blocks[0].attn.k_ln.register_forward_hook(
    make_ln_hook('hook_k_ln_in', 'hook_k_ln_out'))
print("Hooks registered")


Hooks registered


In [5]:
# Forward pass
with torch.no_grad():
    out1 = esm3_original.forward(sequence_tokens=sequence_tokens)
    out2 = esm3_hooked.forward(sequence_tokens=sequence_tokens, return_type="logits")
print("Forward pass completed")
print(f"q_ln: orig_in={'output' in hooks['orig_q_ln_in']}, orig_out={'output' in hooks['orig_q_ln_out']}, hook_in={'output' in hooks['hook_q_ln_in']}, hook_out={'output' in hooks['hook_q_ln_out']}")
print(f"k_ln: orig_in={'output' in hooks['orig_k_ln_in']}, orig_out={'output' in hooks['orig_k_ln_out']}, hook_in={'output' in hooks['hook_k_ln_in']}, hook_out={'output' in hooks['hook_k_ln_out']}")


entered qk_layernorm
entered esm3_use_org_rotary
entered esm3_use_torch_attention_calc
is sequence_id None? True
esm3 or esmc bias: None
entered qk_layernorm
entered esm3_use_org_rotary
entered esm3_use_torch_attention_calc
is sequence_id None? True
esm3 or esmc bias: None
entered qk_layernorm
entered esm3_use_org_rotary
entered esm3_use_torch_attention_calc
is sequence_id None? True
esm3 or esmc bias: None
entered qk_layernorm
entered esm3_use_org_rotary
entered esm3_use_torch_attention_calc
is sequence_id None? True
esm3 or esmc bias: None
entered qk_layernorm
entered esm3_use_org_rotary
entered esm3_use_torch_attention_calc
is sequence_id None? True
esm3 or esmc bias: None
entered qk_layernorm
entered esm3_use_org_rotary
entered esm3_use_torch_attention_calc
is sequence_id None? True
esm3 or esmc bias: None
entered qk_layernorm
entered esm3_use_org_rotary
entered esm3_use_torch_attention_calc
is sequence_id None? True
esm3 or esmc bias: None
entered qk_layernorm
entered esm3_use_org

In [6]:
# Helper function to compare tensors
def compare(name, orig_key, hook_key, rtol=1e-6, atol=1e-6):
    if 'output' not in hooks[orig_key]:
        print(f"{name}: ERROR - {orig_key} not captured")
        return
    if 'output' not in hooks[hook_key]:
        print(f"{name}: ERROR - {hook_key} not captured")
        return
    o = hooks[orig_key]['output']
    h = hooks[hook_key]['output']
    if o.shape != h.shape:
        print(f"{name}: ERROR - shape mismatch: orig={o.shape}, hook={h.shape}")
        return
    if o.dtype != h.dtype: h = h.to(o.dtype)
    diff = torch.abs(o - h)
    max_diff = torch.max(diff).item()
    identical = torch.allclose(o, h, rtol=rtol, atol=atol)
    print(f"{name}: max_diff={max_diff:.8f}, identical={identical}")
    if max_diff > 1e-6:
        pos = torch.unravel_index(torch.argmax(diff), diff.shape)
        print(f"  Max diff at {pos}: orig={o[pos]:.6f}, hook={h[pos]:.6f}, diff={diff[pos]:.6f}")

# Compare all components
compare("LN_QKV vs LN1", 'orig_ln_qkv', 'hook_ln1')
compare("q_ln IN", 'orig_q_ln_in', 'hook_q_ln_in')
compare("q_ln OUT", 'orig_q_ln_out', 'hook_q_ln_out')
compare("k_ln IN", 'orig_k_ln_in', 'hook_k_ln_in')
compare("k_ln OUT", 'orig_k_ln_out', 'hook_k_ln_out')
compare("Rotary Q IN", 'orig_rotary_q_in', 'hook_rotary_q_in')
compare("Rotary K IN", 'orig_rotary_k_in', 'hook_rotary_k_in')
compare("Rotary Q OUT", 'orig_rotary_q_out', 'hook_rotary_q_out')
compare("Rotary K OUT", 'orig_rotary_k_out', 'hook_rotary_k_out')
compare("Attention", 'orig_attn', 'hook_attn')


LN_QKV vs LN1: max_diff=0.00000000, identical=True
q_ln IN: max_diff=0.00000668, identical=False
  Max diff at (tensor(0, device='cuda:0'), tensor(5, device='cuda:0'), tensor(448, device='cuda:0')): orig=9.047671, hook=9.047665, diff=0.000007
q_ln OUT: max_diff=0.00001049, identical=True
  Max diff at (tensor(0, device='cuda:0'), tensor(5, device='cuda:0'), tensor(448, device='cuda:0')): orig=14.376210, hook=14.376200, diff=0.000010
k_ln IN: max_diff=0.00000572, identical=False
  Max diff at (tensor(0, device='cuda:0'), tensor(4, device='cuda:0'), tensor(1059, device='cuda:0')): orig=4.088259, hook=4.088264, diff=0.000006
k_ln OUT: max_diff=0.00000668, identical=False
  Max diff at (tensor(0, device='cuda:0'), tensor(5, device='cuda:0'), tensor(448, device='cuda:0')): orig=9.405515, hook=9.405508, diff=0.000007
Rotary Q IN: max_diff=0.00001049, identical=True
  Max diff at (tensor(0, device='cuda:0'), tensor(5, device='cuda:0'), tensor(7, device='cuda:0'), tensor(0, device='cuda:0')): 

In [7]:
# Compare FFN/MLP outputs
of = hooks['orig_ffn']['output']
hm = hooks['hook_mlp']['output']
if of.dtype != hm.dtype: hm = hm.to(of.dtype)
diff = torch.abs(of - hm)
print(f"FFN/MLP: max_diff={torch.max(diff):.8f}, identical={torch.allclose(of, hm, rtol=1e-6, atol=1e-6)}")
if torch.max(diff) > 1e-6:
    pos = torch.unravel_index(torch.argmax(diff), diff.shape)
    print(f"  Max diff at {pos}: orig={of[pos]:.6f}, hook={hm[pos]:.6f}, diff={diff[pos]:.6f}")


FFN/MLP: max_diff=0.00012207, identical=True
  Max diff at (tensor(0, device='cuda:0'), tensor(5, device='cuda:0'), tensor(1361, device='cuda:0')): orig=-132.116272, hook=-132.116394, diff=0.000122


In [8]:
# Final output comparison
# ESM3 original returns ESMOutput object, hooked returns tensor when return_type="logits"
if hasattr(out1, 'sequence_logits'):
    ol = out1.sequence_logits
else:
    ol = out1
hl = out2
if isinstance(ol, torch.Tensor) and isinstance(hl, torch.Tensor):
    if ol.dtype != hl.dtype: hl = hl.to(ol.dtype)
    diff = torch.abs(ol - hl)
    print(f"Final: max_diff={torch.max(diff):.8f}, within_tol={torch.allclose(ol, hl, rtol=1.3e-6, atol=4e-5)}")
else:
    print(f"Final: Output types - orig={type(ol)}, hook={type(hl)}")

# Cleanup
for h in [h1, h2, h3, h4, h5, h6_orig, h7, h8, h9, h10, h11, h12]: h.remove()


Final: max_diff=0.00000954, within_tol=True
